# 05 - Análise Integrada

Este notebook integra as bases consolidadas do SIH/SUS, CNES e população para construção das análises hospitalares do projeto.

## 1. Configuração inicial

In [1]:
from pathlib import Path
import pandas as pd
pasta_silver = Path("../data/silver")

## 2. Carregamento das bases consolidadas

In [2]:
df_sih = pd.read_parquet( pasta_silver / "sih_multianual.parquet")
df_cnes = pd.read_parquet( pasta_silver / "cnes_multianual.parquet")
df_populacao = pd.read_parquet( pasta_silver / "populacao_multianual.parquet")

print("SIH:", df_sih.shape)
print("CNES:", df_cnes.shape)
print("População:", df_populacao.shape)

SIH: (2286755, 19)
CNES: (198448, 12)
População: (33422, 10)


## 3. Verificação das bases carregadas

In [3]:
print("SIH")
display(df_sih.head())

print("CNES")
display(df_cnes.head())

print("População")
display(df_populacao.head())

SIH


,ANO_CMPT,MES_CMPT,N_AIH,IDENT,SEQ_AIH5,CNES,MUNIC_RES,MUNIC_MOV,DT_INTER,DT_SAIDA,DIAS_PERM,UTI_MES_TO,VAL_TOT,DIAG_PRINC,MORTE,ESPEC,PROC_REA,CAR_INT,ARQUIVO_ORIGEM
0,2021,01,5220103703310,1,000,6665322,521930,521930,2020-10-29,20201101,3,0,485.78,K929,0,03,0303070102,02,RDGO2101.dbc
1,2021,01,5220103703320,1,000,6665322,521930,521930,2020-10-31,20201103,3,0,242.88,R31,0,03,0303150050,02,RDGO2101.dbc
2,2021,01,5220103703353,1,000,6665322,521930,521930,2020-10-08,20201009,1,0,40.38,I743,0,03,0301060070,02,RDGO2101.dbc
3,2021,01,5220103703397,1,000,6665322,521300,521930,2020-10-28,20201102,5,0,503.85,C819,0,03,0304100021,02,RDGO2101.dbc
4,2021,01,5220103703419,1,000,6665322,521930,521930,2020-11-11,20201112,1,0,40.38,K922,1,03,0301060070,02,RDGO2101.dbc


CNES


,CNES,CODUFMUN,TP_UNID,TP_LEITO,CODLEITO,QT_EXIST,QT_SUS,QT_NSUS,COMPETEN,ARQUIVO_ORIGEM,ANO,MES
0,9331603,520010,15,2,33,9,9,0,202101,LTGO2101.dbc,2021,1
1,2335506,520013,05,6,34,4,3,1,202101,LTGO2101.dbc,2021,1
2,2335506,520013,05,1,03,2,1,1,202101,LTGO2101.dbc,2021,1
3,2335506,520013,05,5,45,3,3,0,202101,LTGO2101.dbc,2021,1
4,2335506,520013,05,4,43,3,3,0,202101,LTGO2101.dbc,2021,1


População


,uf,cod_uf,cod_municipio,municipio,populacao,codigo_ibge_7,ano_referencia,ano_publicacao,origem,tipo_dado
0,RO,11,00015,Alta Floresta D'Oeste,22516,1100015,2021,2021.0,IBGE,Estimativa populacional
1,RO,11,00023,Ariquemes,111148,1100023,2021,2021.0,IBGE,Estimativa populacional
2,RO,11,00031,Cabixi,5067,1100031,2021,2021.0,IBGE,Estimativa populacional
3,RO,11,00049,Cacoal,86416,1100049,2021,2021.0,IBGE,Estimativa populacional
4,RO,11,00056,Cerejeiras,16088,1100056,2021,2021.0,IBGE,Estimativa populacional


In [4]:
print("Período SIH:")
print( df_sih["ANO_CMPT"].min(), df_sih["ANO_CMPT"].max())

print("Período CNES:")
print( df_cnes["ANO"].min(), df_cnes["ANO"].max())

print("Anos de população:")
print(
    sorted(
        df_populacao["ano_referencia"]
        .unique()
    )
)

Período SIH:
2021 2026
Período CNES:
2021 2026
Anos de população:
[np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


## 4. Compatibilização dos códigos municipais

Os códigos municipais das fontes possuem formatos diferentes. Antes da integração, é construída e validada uma correspondência entre o código IBGE de sete dígitos e os códigos municipais utilizados pelo DATASUS.

In [5]:
df_populacao["codigo_sus_6"] = (
    df_populacao["codigo_ibge_7"]
    .astype("string")
    .str[:6]
)

In [6]:
dim_municipios = (
    df_populacao[
        [
            "codigo_ibge_7",
            "codigo_sus_6",
            "uf",
            "municipio",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_municipios.head()

,codigo_ibge_7,codigo_sus_6,uf,municipio
0,1100015,110001,RO,Alta Floresta D'Oeste
1,1100023,110002,RO,Ariquemes
2,1100031,110003,RO,Cabixi
3,1100049,110004,RO,Cacoal
4,1100056,110005,RO,Cerejeiras


In [7]:
duplicados_codigo_sus = (
    dim_municipios
    .groupby("codigo_sus_6")
    ["codigo_ibge_7"]
    .nunique()
)

duplicados_codigo_sus = duplicados_codigo_sus[
    duplicados_codigo_sus > 1
]

print(
    "Códigos SUS associados a mais de um código IBGE:",
    len(duplicados_codigo_sus)
)

Códigos SUS associados a mais de um código IBGE: 0


## 5. Validação dos códigos municipais com o CNES

In [8]:
municipios_cnes = set(
    df_cnes["CODUFMUN"]
    .astype("string")
    .str.strip()
    .unique()
)

municipios_dim = set(
    dim_municipios["codigo_sus_6"]
    .dropna()
)

cnes_sem_correspondencia = sorted(
    municipios_cnes
    - municipios_dim
)

print(
    "Municípios do CNES sem correspondência:",
    len(cnes_sem_correspondencia)
)

print(
    cnes_sem_correspondencia[:20]
)

Municípios do CNES sem correspondência: 0
[]


## 6. Validação dos códigos municipais com o SIH/SUS

Os códigos de município de atendimento e de residência presentes no SIH/SUS são comparados com a dimensão municipal para verificar a correspondência com os códigos utilizados pelo IBGE.

In [9]:
municipios_mov_sih = set(
    df_sih["MUNIC_MOV"]
    .astype("string")
    .str.strip()
    .dropna()
    .unique()
)

municipios_res_sih = set(
    df_sih["MUNIC_RES"]
    .astype("string")
    .str.strip()
    .dropna()
    .unique()
)

mov_sem_correspondencia = sorted(
    municipios_mov_sih - municipios_dim
)

res_sem_correspondencia = sorted(
    municipios_res_sih - municipios_dim
)

print(
    "Municípios de atendimento sem correspondência:",
    len(mov_sem_correspondencia)
)

print(
    "Municípios de residência sem correspondência:",
    len(res_sem_correspondencia)
)

Municípios de atendimento sem correspondência: 0
Municípios de residência sem correspondência: 0


In [10]:
print("MUNIC_MOV sem correspondência:")
print(mov_sem_correspondencia[:20])

print("\nMUNIC_RES sem correspondência:")
print(res_sem_correspondencia[:20])

MUNIC_MOV sem correspondência:
[]

MUNIC_RES sem correspondência:
[]


## 7. Preparação temporal do SIH/SUS

As datas de internação e saída são convertidas para formato de data para permitir análises temporais com base no período efetivo da internação.

In [11]:
df_sih["DT_INTER_DATA"] = pd.to_datetime(
    df_sih["DT_INTER"],
    errors="coerce",
    format="mixed"
)

df_sih["DT_SAIDA_DATA"] = pd.to_datetime(
    df_sih["DT_SAIDA"],
    errors="coerce",
    format="mixed"
)

print("Datas de internação inválidas:", df_sih["DT_INTER_DATA"].isna().sum())
print("Datas de saída inválidas:", df_sih["DT_SAIDA_DATA"].isna().sum())

print(
    "\nPeríodo das internações:",
    df_sih["DT_INTER_DATA"].min(),
    "até",
    df_sih["DT_INTER_DATA"].max()
)

print(
    "Período das saídas:",
    df_sih["DT_SAIDA_DATA"].min(),
    "até",
    df_sih["DT_SAIDA_DATA"].max()
)

Datas de internação inválidas: 0
Datas de saída inválidas: 0

Período das internações: 2008-01-01 00:00:00 até 2026-06-30 00:00:00
Período das saídas: 2020-08-03 00:00:00 até 2026-06-30 00:00:00


In [12]:
df_sih["ANO_INTER"] = df_sih["DT_INTER_DATA"].dt.year
df_sih["MES_INTER"] = df_sih["DT_INTER_DATA"].dt.month

df_sih["COMP_INTER"] = (
    df_sih["DT_INTER_DATA"]
    .dt.to_period("M")
)

df_sih[
    [
        "ANO_CMPT",
        "MES_CMPT",
        "DT_INTER_DATA",
        "DT_SAIDA_DATA",
        "ANO_INTER",
        "MES_INTER",
        "COMP_INTER"
    ]
].head()

,ANO_CMPT,MES_CMPT,DT_INTER_DATA,DT_SAIDA_DATA,ANO_INTER,MES_INTER,COMP_INTER
0,2021,01,2020-10-29,2020-11-01,2020,10,2020-10
1,2021,01,2020-10-31,2020-11-03,2020,10,2020-10
2,2021,01,2020-10-08,2020-10-09,2020,10,2020-10
3,2021,01,2020-10-28,2020-11-02,2020,10,2020-10
4,2021,01,2020-11-11,2020-11-12,2020,11,2020-11


### 7.1. Investigação das internações anteriores ao período analisado

Foram identificadas datas de internação anteriores ao período principal da análise. Os registros serão avaliados antes de qualquer decisão de filtragem.

In [13]:
internacoes_antigas = df_sih[
    df_sih["DT_INTER_DATA"] < "2020-01-01"
].copy()

print(
    "Internações anteriores a 2020:",
    len(internacoes_antigas)
)

print("\nPor ano de internação:")
print(
    internacoes_antigas["ANO_INTER"]
    .value_counts()
    .sort_index()
)

Internações anteriores a 2020: 13365

Por ano de internação:
ANO_INTER
2008    4348
2009     399
2010     129
2011     336
2012     763
2013     453
2014     493
2015     820
2016     905
2017    1084
2018    2040
2019    1595
Name: count, dtype: int64


In [14]:
print("Tipos de AIH nos registros antigos:")
print( internacoes_antigas["IDENT"].value_counts(dropna=False))

internacoes_antigas[
    [
        "ANO_CMPT",
        "MES_CMPT",
        "N_AIH",
        "IDENT",
        "SEQ_AIH5",
        "CNES",
        "DT_INTER_DATA",
        "DT_SAIDA_DATA",
        "DIAS_PERM"
    ]
].sort_values("DT_INTER_DATA").head(20)

Tipos de AIH nos registros antigos:
IDENT
5    13365
Name: count, dtype: Int64


,ANO_CMPT,MES_CMPT,N_AIH,IDENT,SEQ_AIH5,CNES,DT_INTER_DATA,DT_SAIDA_DATA,DIAS_PERM
1301,2021,01,5208100143180,5,000,2535939,2008-01-01,2021-01-31,31
660377,2022,11,5208100142795,5,000,2535939,2008-01-01,2022-11-30,30
660376,2022,11,5208100142784,5,000,2535939,2008-01-01,2022-11-30,30
660375,2022,11,5208100142707,5,000,2535939,2008-01-01,2022-11-30,30
660374,2022,11,5208100142663,5,000,2535939,2008-01-01,2022-11-30,30
660373,2022,11,5208100142652,5,000,2535939,2008-01-01,2022-11-30,30
660354,2022,11,5208100144588,5,000,2535939,2008-01-01,2022-11-30,30
660353,2022,11,5208100144533,5,000,2535939,2008-01-01,2022-11-30,30
660307,2022,11,5208100144313,5,000,2535939,2008-01-01,2022-11-30,30
660306,2022,11,5208100144148,5,000,2535939,2008-01-01,2022-11-30,30


In [15]:
df_sih["DIAS_CALCULADOS"] = (
    df_sih["DT_SAIDA_DATA"]
    - df_sih["DT_INTER_DATA"]
).dt.days

df_sih[
    [
        "DT_INTER_DATA",
        "DT_SAIDA_DATA",
        "DIAS_PERM",
        "DIAS_CALCULADOS",
        "IDENT",
        "SEQ_AIH5"
    ]
].sort_values(
    "DT_INTER_DATA"
).head(20)

,DT_INTER_DATA,DT_SAIDA_DATA,DIAS_PERM,DIAS_CALCULADOS,IDENT,SEQ_AIH5
171991,2008-01-01,2021-07-31,31,4960,5,000
1151842,2008-01-01,2024-01-31,31,5874,5,000
753362,2008-01-01,2023-02-28,28,5537,5,000
2065341,2008-01-01,2026-01-31,31,6605,5,000
2065342,2008-01-01,2026-01-31,31,6605,5,000
2065343,2008-01-01,2026-01-31,31,6605,5,000
2065344,2008-01-01,2026-01-31,31,6605,5,000
1328791,2008-01-01,2024-06-30,30,6025,5,000
1328792,2008-01-01,2024-06-30,30,6025,5,000
1328793,2008-01-01,2024-06-30,30,6025,5,000


### 7.2. Tratamento das AIHs de longa permanência

A investigação mostrou que todas as internações anteriores a 2020 correspondem a AIHs do tipo 5, relacionadas a longa permanência.

Nesses registros, a data de internação pode representar o início original da internação, enquanto o registro mensal representa a continuidade da permanência. Por isso, essas AIHs não serão contabilizadas como novas internações com base em `DT_INTER`.

As AIHs de longa permanência são mantidas separadamente para eventual análise específica desse tipo de registro. Elas não são contabilizadas como novas internações e não são incluídas na referência de permanência das AIHs regulares utilizada nas análises posteriores.

In [16]:
df_sih["AIH_LONGA_PERMANENCIA"] = (
    df_sih["IDENT"]
    .astype("string")
    .str.strip()
    .eq("5")
)

print(
    "AIHs regulares:",
    (~df_sih["AIH_LONGA_PERMANENCIA"]).sum()
)

print(
    "AIHs de longa permanência:",
    df_sih["AIH_LONGA_PERMANENCIA"].sum()
)

print(
    "\nPercentual de longa permanência:",
    round(
        df_sih["AIH_LONGA_PERMANENCIA"].mean() * 100,
        2
    ),
    "%"
)

AIHs regulares: 2235030
AIHs de longa permanência: 51725

Percentual de longa permanência: 2.26 %


In [17]:
df_sih["DIAS_PERM_NUM"] = pd.to_numeric(
    df_sih["DIAS_PERM"],
    errors="coerce"
)

print("DIAS_PERM inválidos:", df_sih["DIAS_PERM_NUM"].isna().sum())

DIAS_PERM inválidos: 0


In [18]:
resumo_ident = (
    df_sih
    .groupby("IDENT", dropna=False)
    .agg(
        registros=("N_AIH", "size"),
        data_inter_min=("DT_INTER_DATA", "min"),
        data_inter_max=("DT_INTER_DATA", "max"),
        mediana_dias_perm=("DIAS_PERM_NUM", "median")
    )
    .reset_index()
)

resumo_ident

,IDENT,registros,data_inter_min,data_inter_max,mediana_dias_perm
0,1,2235030,2020-06-18,2026-06-30,2.0
1,5,51725,2008-01-01,2026-05-31,30.0


### 7.3. Definição do recorte temporal da demanda

Para as análises de novas internações são consideradas apenas as AIHs regulares (`IDENT = 1`) cuja data de internação pertence ao período de janeiro de 2021 a junho de 2026.

As AIHs de longa permanência (`IDENT = 5`) são mantidas separadamente, pois representam continuidade de internações e não devem ser contabilizadas como novas admissões.

In [19]:
df_sih_regulares = df_sih[~df_sih["AIH_LONGA_PERMANENCIA"]].copy()

df_sih_longa = df_sih[df_sih["AIH_LONGA_PERMANENCIA"]].copy()

inicio_analise = pd.Timestamp("2021-01-01")
fim_analise = pd.Timestamp("2026-06-30")

df_sih_demanda = df_sih_regulares[
    df_sih_regulares["DT_INTER_DATA"].between(
        inicio_analise,
        fim_analise
    )
].copy()

print("AIHs regulares totais:", len(df_sih_regulares))
print("AIHs regulares no período:", len(df_sih_demanda))
print(
    "AIHs regulares anteriores a 2021:",
    (df_sih_regulares["DT_INTER_DATA"] < inicio_analise).sum()
)

print("AIHs de longa permanência:", len(df_sih_longa))

AIHs regulares totais: 2235030
AIHs regulares no período: 2217182
AIHs regulares anteriores a 2021: 17848
AIHs de longa permanência: 51725


In [20]:
resumo_demanda_ano = (
    df_sih_demanda
    .assign(
        ano_internacao=df_sih_demanda["DT_INTER_DATA"].dt.year
    )
    .groupby("ano_internacao")
    .size()
    .reset_index(name="internacoes")
)

resumo_demanda_ano

,ano_internacao,internacoes
0,2021,337659
1,2022,364479
2,2023,415544
3,2024,440438
4,2025,459420
5,2026,199642


## 8. Integração temporal entre SIH/SUS e CNES

Os registros de internações são relacionados as informações mensais de leitos do CNES por estabelecimento e competência.

Como o CNES possui registros por tipo de leito, inicialmente os leitos são agregados por estabelecimento e mês para obter a capacidade hospitalar mensal.

In [21]:
df_cnes["QT_EXIST_NUM"] = pd.to_numeric( df_cnes["QT_EXIST"], errors="coerce")
df_cnes["QT_SUS_NUM"] = pd.to_numeric( df_cnes["QT_SUS"], errors="coerce")
df_cnes["QT_NSUS_NUM"] = pd.to_numeric( df_cnes["QT_NSUS"], errors="coerce")

df_cnes["COMP_CNES"] = pd.PeriodIndex.from_fields(
    year=df_cnes["ANO"].astype(int),
    month=df_cnes["MES"].astype(int),
    freq="M"
)

print(
    "Período CNES:",
    df_cnes["COMP_CNES"].min(),
    "até",
    df_cnes["COMP_CNES"].max()
)

Período CNES: 2021-01 até 2026-07


In [22]:
capacidade_hospital_mes = (
    df_cnes
    .groupby(
        [
            "CNES",
            "CODUFMUN",
            "COMP_CNES"
        ],
        as_index=False
    )
    .agg(
        leitos_existentes=("QT_EXIST_NUM", "sum"),
        leitos_sus=("QT_SUS_NUM", "sum"),
        leitos_nao_sus=("QT_NSUS_NUM", "sum")
    )
)

capacidade_hospital_mes.head()

,CNES,CODUFMUN,COMP_CNES,leitos_existentes,leitos_sus,leitos_nao_sus
0,0024074,520870,2021-01,228,130,98
1,0024074,520870,2021-02,228,183,45
2,0024074,520870,2021-03,204,181,23
3,0024074,520870,2021-04,204,181,23
4,0024074,520870,2021-05,204,181,23


In [23]:
print( "Registros hospital-mês:", len(capacidade_hospital_mes))
print( "Hospitais distintos:", capacidade_hospital_mes["CNES"].nunique())

print( "Competências:", capacidade_hospital_mes["COMP_CNES"].nunique())

print(
    "Período:",
    capacidade_hospital_mes["COMP_CNES"].min(),
    "até",
    capacidade_hospital_mes["COMP_CNES"].max()
)

print(
    "Inconsistências de leitos:",
    (
        capacidade_hospital_mes["leitos_existentes"]
        != (
            capacidade_hospital_mes["leitos_sus"]
            + capacidade_hospital_mes["leitos_nao_sus"]
        )
    ).sum()
)

Registros hospital-mês: 34199
Hospitais distintos: 585
Competências: 67
Período: 2021-01 até 2026-07
Inconsistências de leitos: 0


### 8.1. Validação das AIHs regulares

Antes da agregação da demanda hospitalar, é verificada a ocorrência de números de AIH repetidos no conjunto de internações regulares selecionado para o período analisado.

In [24]:
duplicados_aih_demanda = (
    df_sih_demanda["N_AIH"]
    .duplicated(keep=False)
)

print(
    "Registros com N_AIH repetido:",
    duplicados_aih_demanda.sum()
)

print(
    "Números de AIH repetidos:",
    df_sih_demanda.loc[
        duplicados_aih_demanda,
        "N_AIH"
    ].nunique()
)

Registros com N_AIH repetido: 0
Números de AIH repetidos: 0


In [25]:
aih_repetidas = (
    df_sih_demanda.loc[
        duplicados_aih_demanda,
        [
            "N_AIH",
            "CNES",
            "MUNIC_MOV",
            "DT_INTER_DATA",
            "DT_SAIDA_DATA",
            "DIAS_PERM_NUM",
            "ANO_CMPT",
            "MES_CMPT"
        ]
    ]
    .sort_values(
        [
            "N_AIH",
            "ANO_CMPT",
            "MES_CMPT"
        ]
    )
)

aih_repetidas.head(20)

,N_AIH,CNES,MUNIC_MOV,DT_INTER_DATA,DT_SAIDA_DATA,DIAS_PERM_NUM,ANO_CMPT,MES_CMPT


### 8.2. Agregação mensal da demanda hospitalar

As internações regulares são agregadas por estabelecimento e mês de internação. Como não foram identificados números de AIH repetidos no período analisado, cada registro é contabilizado como uma internação.

In [26]:
df_sih_demanda["CNES"] = (
    df_sih_demanda["CNES"]
    .astype("string")
    .str.strip()
    .str.zfill(7)
)

capacidade_hospital_mes["CNES"] = (
    capacidade_hospital_mes["CNES"]
    .astype("string")
    .str.strip()
    .str.zfill(7)
)

In [27]:
demanda_hospital_mes = (
    df_sih_demanda
    .groupby(
        [
            "CNES",
            "MUNIC_MOV",
            "COMP_INTER"
        ],
        as_index=False
    )
    .agg(
        internacoes=("N_AIH", "size")
    )
)

demanda_hospital_mes.head()

,CNES,MUNIC_MOV,COMP_INTER,internacoes
0,0024074,520870,2021-01,289
1,0024074,520870,2021-02,290
2,0024074,520870,2021-03,438
3,0024074,520870,2021-04,416
4,0024074,520870,2021-05,383


In [28]:
print( "Registros hospital-mês com internações:", len(demanda_hospital_mes))
print( "Hospitais com internações:", demanda_hospital_mes["CNES"].nunique())
print( "Total de internações:", demanda_hospital_mes["internacoes"].sum())

print(
    "Período da demanda:",
    demanda_hospital_mes["COMP_INTER"].min(),
    "até",
    demanda_hospital_mes["COMP_INTER"].max()
)

Registros hospital-mês com internações: 16031
Hospitais com internações: 303
Total de internações: 2217182
Período da demanda: 2021-01 até 2026-06


### 8.3. Integração entre demanda e capacidade

A demanda mensal de internações é relacionada à capacidade de leitos cadastrada no CNES para o mesmo estabelecimento e competência.

In [29]:
demanda_capacidade = demanda_hospital_mes.merge(
    capacidade_hospital_mes,
    left_on=[
        "CNES",
        "COMP_INTER"
    ],
    right_on=[
        "CNES",
        "COMP_CNES"
    ],
    how="left",
    validate="one_to_one",
    indicator=True
)

demanda_capacidade.head()

,CNES,MUNIC_MOV,COMP_INTER,internacoes,CODUFMUN,COMP_CNES,leitos_existentes,leitos_sus,leitos_nao_sus,_merge
0,0024074,520870,2021-01,289,520870,2021-01,228,130,98,both
1,0024074,520870,2021-02,290,520870,2021-02,228,183,45,both
2,0024074,520870,2021-03,438,520870,2021-03,204,181,23,both
3,0024074,520870,2021-04,416,520870,2021-04,204,181,23,both
4,0024074,520870,2021-05,383,520870,2021-05,204,181,23,both


In [30]:
print(demanda_capacidade["_merge"].value_counts())
print(
    "\nRegistros sem capacidade CNES:",
    (
        demanda_capacidade["_merge"]
        != "both"
    ).sum()
)

print(
    "Hospitais sem correspondência CNES:",
    demanda_capacidade.loc[
        demanda_capacidade["_merge"] != "both",
        "CNES"
    ].nunique()
)

_merge
both          16030
left_only         1
right_only        0
Name: count, dtype: int64

Registros sem capacidade CNES: 1
Hospitais sem correspondência CNES: 1


### 8.4. Investigação dos registros sem correspondência no CNES

Os registros de demanda sem capacidade correspondente no CNES são avaliados individualmente antes da definição do conjunto integrado final.

In [31]:
sem_capacidade = demanda_capacidade[
    demanda_capacidade["_merge"] == "left_only"
].copy()

sem_capacidade[
    [
        "CNES",
        "MUNIC_MOV",
        "COMP_INTER",
        "internacoes"
    ]
]

,CNES,MUNIC_MOV,COMP_INTER,internacoes
10759,2535181,520425,2022-06,1


In [32]:
cnes_sem_correspondencia = (
    sem_capacidade["CNES"]
    .iloc[0]
)

capacidade_hospital_mes[
    capacidade_hospital_mes["CNES"]
    == cnes_sem_correspondencia
].sort_values("COMP_CNES")

,CNES,CODUFMUN,COMP_CNES,leitos_existentes,leitos_sus,leitos_nao_sus
18877,2535181,520425,2021-01,18,18,0
18878,2535181,520425,2021-02,18,18,0
18879,2535181,520425,2021-03,18,18,0
18880,2535181,520425,2021-04,18,18,0
18881,2535181,520425,2021-05,18,18,0
...,...,...,...,...,...,...
18938,2535181,520425,2026-03,18,18,0
18939,2535181,520425,2026-04,18,18,0
18940,2535181,520425,2026-05,18,18,0
18941,2535181,520425,2026-06,18,18,0


In [33]:
municipios_divergentes = demanda_capacidade[
    (demanda_capacidade["_merge"] == "both")
    &
    (
        demanda_capacidade["MUNIC_MOV"].astype("string")
        !=
        demanda_capacidade["CODUFMUN"].astype("string")
    )
]

print("Registros com município divergente:",len(municipios_divergentes))

Registros com município divergente: 0


### 8.5. Definição do conjunto integrado para análise de capacidade

A integração entre SIH/SUS e CNES apresentou correspondência para 16.030 das 16.031 combinações hospital-mês com internações.

Foi identificado apenas um registro sem informação de capacidade correspondente no CNES, referente ao estabelecimento 2535181, na competência junho de 2022, com uma internação.

Esse registro é mantido nas análises de demanda hospitalar, mas não é utilizado nas análises que relacionam demanda e capacidade, evitando a imputação de uma quantidade de leitos não observada na base.

In [34]:
demanda_capacidade_validada = (
    demanda_capacidade[
        demanda_capacidade["_merge"] == "both"
    ]
    .drop(columns="_merge")
    .copy()
)

print( "Registros integrados:",len(demanda_capacidade_validada))
print( "Internações com capacidade correspondente:", demanda_capacidade_validada["internacoes"].sum())
print( "Internações sem capacidade correspondente:", sem_capacidade["internacoes"].sum())

Registros integrados: 16030
Internações com capacidade correspondente: 2217181
Internações sem capacidade correspondente: 1


In [35]:
cobertura_integracao = (
    demanda_capacidade_validada["internacoes"].sum()
    / demanda_hospital_mes["internacoes"].sum()
    * 100
)

print( f"Cobertura da integração SIH/CNES: "f"{cobertura_integracao:.6f}%")

Cobertura da integração SIH/CNES: 99.999955%


## 9. Integração com a população municipal

A população municipal é incorporada às análises utilizando o código municipal compatível com o DATASUS.

A base populacional cobre o período de 2021 a 2026. Para 2021, 2024, 2025 e 2026 são utilizadas estimativas populacionais do IBGE. Para 2022 é utilizada a população de referência publicada pelo IBGE para o TCU, baseada no Censo 2022 e na Malha 2023.

Para 2023 é utilizada uma população de referência derivada por interpolação linear entre os valores municipais de 2022 e 2024. Esse valor é utilizado apenas para fins analíticos e não representa uma estimativa oficial publicada pelo IBGE.

In [36]:
df_populacao = pd.read_parquet( pasta_silver / "populacao_multianual.parquet")

df_populacao["codigo_sus_6"] = (
    df_populacao["codigo_ibge_7"]
    .astype("string")
    .str[:6]
)

print("Dimensão:", df_populacao.shape)

print(
    "Anos disponíveis:",
    sorted(
        df_populacao["ano_referencia"]
        .astype(int)
        .unique()
    )
)

Dimensão: (33422, 11)
Anos disponíveis: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


In [37]:
populacao_go = (
    df_populacao[
        df_populacao["uf"] == "GO"
    ]
    .copy()
)

validacao_populacao_go = (
    populacao_go
    .groupby("ano_referencia")
    .agg(
        municipios=("codigo_sus_6", "nunique"),
        populacao_total=("populacao", "sum")
    )
    .reset_index()
)

validacao_populacao_go

,ano_referencia,municipios,populacao_total
0,2021,246,7206589
1,2022,246,7056495
2,2023,246,7203492
3,2024,246,7350483
4,2025,246,7423629
5,2026,246,7495033


### 9.1. Internações por município de residência

Para indicadores relacionados à população residente, as internações são associadas ao município de residência do paciente (`MUNIC_RES`).

Nesta etapa são consideradas apenas as internações de residentes em municípios de Goiás. Registros de pacientes residentes em outras unidades da federação permanecem na base hospitalar, mas não são utilizados no cálculo de indicadores populacionais dos municípios goianos.

In [38]:
# Padroniza o código do município de residência
df_sih_demanda["MUNIC_RES"] = (
    df_sih_demanda["MUNIC_RES"]
    .astype("string")
    .str.zfill(6)
)

# Códigos dos municípios de Goiás
codigos_go = set( populacao_go["codigo_sus_6"].dropna().unique())

# Internações de residentes em Goiás
demanda_residentes_go = (
    df_sih_demanda[
        df_sih_demanda["MUNIC_RES"].isin(codigos_go)
    ]
    .copy()
)

print( "Internações totais na base de demanda:", len(df_sih_demanda))
print( "Internações de residentes em Goiás:",len(demanda_residentes_go))
print( "Internações de residentes fora de Goiás:", len(df_sih_demanda) - len(demanda_residentes_go))

Internações totais na base de demanda: 2217182
Internações de residentes em Goiás: 2199260
Internações de residentes fora de Goiás: 17922


In [39]:
internacoes_residencia_ano = (
    demanda_residentes_go
    .groupby(
        [
            "ANO_INTER",
            "MUNIC_RES"
        ]
    )
    .agg(
        internacoes=("N_AIH", "size")
    )
    .reset_index()
)

print( "Registros município-ano:", len(internacoes_residencia_ano))
internacoes_residencia_ano.head()

Registros município-ano: 1476


,ANO_INTER,MUNIC_RES,internacoes
0,2021,520005,796
1,2021,520010,495
2,2021,520013,1015
3,2021,520015,79
4,2021,520017,85


In [40]:
resumo_internacoes_residencia = (
    internacoes_residencia_ano
    .groupby("ANO_INTER")
    .agg(
        municipios=("MUNIC_RES", "nunique"),
        internacoes=("internacoes", "sum")
    )
    .reset_index()
)

resumo_internacoes_residencia

,ANO_INTER,municipios,internacoes
0,2021,246,334351
1,2022,246,361846
2,2023,246,412780
3,2024,246,437359
4,2025,246,455189
5,2026,246,197735


In [41]:
base_residencia_populacao = (
    populacao_go[
        [
            "codigo_sus_6",
            "municipio",
            "ano_referencia",
            "populacao",
            "origem",
            "tipo_dado"
        ]
    ]
    .copy()
)

base_residencia_populacao["ano_referencia"] = (
    base_residencia_populacao["ano_referencia"]
    .astype(int)
)

internacoes_residencia_ano["ANO_INTER"] = (
    internacoes_residencia_ano["ANO_INTER"]
    .astype(int)
)

In [42]:
base_residencia_populacao.head()

,codigo_sus_6,municipio,ano_referencia,populacao,origem,tipo_dado
5323,520005,Abadia de Goiás,2021,9158,IBGE,Estimativa populacional
5324,520010,Abadiânia,2021,20873,IBGE,Estimativa populacional
5325,520013,Acreúna,2021,22710,IBGE,Estimativa populacional
5326,520015,Adelândia,2021,2515,IBGE,Estimativa populacional
5327,520017,Água Fria de Goiás,2021,5843,IBGE,Estimativa populacional


In [43]:
indicadores_residencia = (
    base_residencia_populacao
    .merge(
        internacoes_residencia_ano,
        left_on=[
            "ano_referencia",
            "codigo_sus_6"
        ],
        right_on=[
            "ANO_INTER",
            "MUNIC_RES"
        ],
        how="left",
        validate="one_to_one"
    )
)

indicadores_residencia["internacoes"] = (
    indicadores_residencia["internacoes"]
    .fillna(0)
    .astype(int)
)

print( "Registros após integração:", len(indicadores_residencia))
print( "Populações nulas:", indicadores_residencia["populacao"].isna().sum())
print( "Internações nulas:", indicadores_residencia["internacoes"].isna().sum())

Registros após integração: 1476
Populações nulas: 0
Internações nulas: 0


### 9.2. Internações por população residente

A quantidade de internações é relacionada à população do município de residência para permitir comparações entre municípios com diferentes tamanhos populacionais.

O indicador representa o número de internações por 1.000 habitantes no período disponível. Para os anos de 2021 a 2025, o período corresponde ao ano completo. Em 2026, os dados de internações abrangem apenas janeiro a junho e, portanto, o indicador é parcial e não deve ser comparado diretamente aos anos completos.

In [44]:
indicadores_residencia["internacoes_por_1000_hab"] = (
    indicadores_residencia["internacoes"]
    / indicadores_residencia["populacao"]
    * 1000
)

indicadores_residencia["periodo_completo"] = ( indicadores_residencia["ano_referencia"] < 2026)

indicadores_residencia[
    [
        "ano_referencia",
        "codigo_sus_6",
        "municipio",
        "populacao",
        "internacoes",
        "internacoes_por_1000_hab",
        "periodo_completo"
    ]
].head()

,ano_referencia,codigo_sus_6,municipio,populacao,internacoes,internacoes_por_1000_hab,periodo_completo
0,2021,520005,Abadia de Goiás,9158,796,86.918541,True
1,2021,520010,Abadiânia,20873,495,23.714847,True
2,2021,520013,Acreúna,22710,1015,44.693967,True
3,2021,520015,Adelândia,2515,79,31.411531,True
4,2021,520017,Água Fria de Goiás,5843,85,14.547322,True


In [45]:
resumo_indicadores_residencia = (
    indicadores_residencia
    .groupby("ano_referencia")
    .agg(
        municipios=("codigo_sus_6", "nunique"),
        populacao_total=("populacao", "sum"),
        internacoes=("internacoes", "sum")
    )
    .reset_index()
)

resumo_indicadores_residencia["internacoes_por_1000_hab"] = (
    resumo_indicadores_residencia["internacoes"]
    / resumo_indicadores_residencia["populacao_total"]
    * 1000
)

resumo_indicadores_residencia

,ano_referencia,municipios,populacao_total,internacoes,internacoes_por_1000_hab
0,2021,246,7206589,334351,46.395181
1,2022,246,7056495,361846,51.278432
2,2023,246,7203492,412780,57.302764
3,2024,246,7350483,437359,59.500716
4,2025,246,7423629,455189,61.316238
5,2026,246,7495033,197735,26.382139


### 9.3. Validação dos indicadores populacionais

A integração entre internações e população resultou em informações para os 246 municípios de Goiás em todos os anos analisados.

O indicador de internações por 1.000 habitantes utiliza o município de residência do paciente como referência. Para 2021 a 2025, os valores representam anos completos. Em 2026, o indicador considera apenas as internações disponíveis entre janeiro e junho e deve ser interpretado como parcial.

Para 2023, o denominador populacional corresponde à população de referência derivada por interpolação linear entre 2022 e 2024.

In [46]:
resumo_indicadores_residencia["periodo"] = (
    resumo_indicadores_residencia["ano_referencia"]
    .apply(
        lambda ano: (
            "Janeiro a junho"
            if ano == 2026
            else "Ano completo"
        )
    )
)

resumo_indicadores_residencia["tipo_populacao"] = (
    resumo_indicadores_residencia["ano_referencia"]
    .apply(
        lambda ano: (
            "População derivada"
            if ano == 2023
            else "População IBGE"
        )
    )
)

resumo_indicadores_residencia

,ano_referencia,municipios,populacao_total,internacoes,internacoes_por_1000_hab,periodo,tipo_populacao
0,2021,246,7206589,334351,46.395181,Ano completo,População IBGE
1,2022,246,7056495,361846,51.278432,Ano completo,População IBGE
2,2023,246,7203492,412780,57.302764,Ano completo,População derivada
3,2024,246,7350483,437359,59.500716,Ano completo,População IBGE
4,2025,246,7423629,455189,61.316238,Ano completo,População IBGE
5,2026,246,7495033,197735,26.382139,Janeiro a junho,População IBGE


## 10. Demanda hospitalar por município de residência

### 10.1. Municípios com maior número de internações em 2025

Nesta etapa é analisada a distribuição das internações entre os municípios de residência dos pacientes.

O número absoluto de internações permite identificar os municípios que concentram maior demanda hospitalar entre seus residentes. Para evitar a comparação entre um ano completo e um período parcial, o ranking utiliza inicialmente 2025, último ano com dados completos de internações.

O indicador por 1.000 habitantes será utilizado de forma complementar, permitindo considerar diferenças no tamanho da população municipal.

In [47]:
demanda_municipal_2025 = (
    indicadores_residencia[
        indicadores_residencia["ano_referencia"] == 2025
    ]
    .sort_values(
        "internacoes",
        ascending=False
    )
    .reset_index(drop=True)
)

demanda_municipal_2025[
    [
        "municipio",
        "populacao",
        "internacoes",
        "internacoes_por_1000_hab"
    ]
].head(10)

,municipio,populacao,internacoes,internacoes_por_1000_hab
0,Goiânia,1503256,79940,53.177902
1,Aparecida de Goiânia,556021,35861,64.495765
2,Anápolis,420300,24608,58.548656
3,Rio Verde,241494,18304,75.794844
4,Trindade,153560,9954,64.821568
5,Senador Canedo,175042,9832,56.169376
6,Itumbiara,113322,8899,78.528441
7,Formosa,121559,7781,64.010069
8,Goianira,81495,6752,82.851709
9,Luziânia,221262,6654,30.072945


In [48]:
print( "Municípios analisados:", demanda_municipal_2025["codigo_sus_6"].nunique())
print( "Total de internações em 2025:", demanda_municipal_2025["internacoes"].sum())

Municípios analisados: 246
Total de internações em 2025: 455189


### 10.2. Internações por 1.000 habitantes em 2025

O número absoluto de internações tende a ser maior nos municípios mais populosos. Por isso, também é analisado o número de internações por 1.000 habitantes, utilizando a população residente como referência.

Esse indicador permite comparar a frequência de internações entre municípios de diferentes tamanhos populacionais.

In [49]:
demanda_relativa_2025 = (
    indicadores_residencia[
        indicadores_residencia["ano_referencia"] == 2025
    ]
    .sort_values(
        "internacoes_por_1000_hab",
        ascending=False
    )
    .reset_index(drop=True)
)

demanda_relativa_2025[
    [
        "municipio",
        "populacao",
        "internacoes",
        "internacoes_por_1000_hab"
    ]
].head(10)

,municipio,populacao,internacoes,internacoes_por_1000_hab
0,São Miguel do Araguaia,22040,6584,298.729583
1,Mundo Novo,6202,1221,196.871977
2,Paranaiguara,7356,1368,185.970636
3,Campinaçu,3760,637,169.414894
4,Damolândia,2753,456,165.637486
5,Heitoraí,3342,542,162.178336
6,Mutunópolis,3540,521,147.175141
7,Silvânia,23150,3391,146.479482
8,Caiapônia,16628,2433,146.319461
9,Arenópolis,2904,418,143.939394


In [50]:
print( "Municípios analisados:", demanda_relativa_2025["codigo_sus_6"].nunique())
print( "Menor população entre os 10 primeiros:", demanda_relativa_2025.head(10)["populacao"].min())

Municípios analisados: 246
Menor população entre os 10 primeiros: 2753


### 10.3. Persistência da demanda relativa entre 2021 e 2025

Taxas elevadas em um único ano podem ser influenciadas por variações no número de internações, especialmente em municípios com menor população.

Por isso, é analisado o comportamento das internações por 1.000 habitantes entre 2021 e 2025, considerando apenas os anos completos disponíveis.

In [51]:
demanda_relativa_2021_2025 = (
    indicadores_residencia[
        indicadores_residencia["ano_referencia"].between(
            2021,
            2025
        )
    ]
    .groupby(
        [
            "codigo_sus_6",
            "municipio"
        ]
    )
    .agg(
        populacao_media=("populacao", "mean"),
        internacoes_total=("internacoes", "sum"),
        taxa_media_anual_1000=(
            "internacoes_por_1000_hab",
            "mean"
        ),
        taxa_minima_1000=(
            "internacoes_por_1000_hab",
            "min"
        ),
        taxa_maxima_1000=(
            "internacoes_por_1000_hab",
            "max"
        ),
        anos_analisados=(
            "ano_referencia",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        "taxa_media_anual_1000",
        ascending=False
    )
)

demanda_relativa_2021_2025[
    [
        "municipio",
        "populacao_media",
        "internacoes_total",
        "taxa_media_anual_1000",
        "taxa_minima_1000",
        "taxa_maxima_1000",
        "anos_analisados"
    ]
].head(10)

,municipio,populacao_media,internacoes_total,taxa_media_anual_1000,taxa_minima_1000,taxa_maxima_1000,anos_analisados
108,Indiara,17231.8,16399,191.505009,71.337225,268.501494,5
153,Mundo Novo,5872.0,5231,179.244671,113.427048,205.045798,5
219,São Miguel do Araguaia,21971.6,17303,157.221896,80.502283,298.729583,5
50,Campinaçu,3717.4,2910,156.391683,116.041896,190.938511,5
177,Paranaiguara,8038.4,5778,147.365641,96.27238,185.970636,5
103,Heitoraí,3431.0,2352,137.536261,119.1876,162.178336,5
234,Turvânia,4482.4,2787,124.387311,90.36677,164.545658,5
13,Americano do Brasil,5458.0,3332,122.095647,84.220532,147.176269,5
76,Damolândia,2783.2,1655,119.619719,67.595109,165.637486,5
78,Diorama,2119.0,1233,118.808938,61.768268,173.999011,5


In [52]:
print( "Municípios analisados:", demanda_relativa_2021_2025["codigo_sus_6"].nunique())

print( "Municípios com 5 anos disponíveis:",
    (
        demanda_relativa_2021_2025["anos_analisados"]
        == 5
    ).sum()
)

Municípios analisados: 246
Municípios com 5 anos disponíveis: 246


### 10.4. Síntese da demanda municipal

A análise mostra que os municípios com maior número absoluto de internações não são necessariamente os mesmos que apresentam as maiores taxas por população.

Municípios mais populosos, como Goiânia, Aparecida de Goiânia e Anápolis, concentram grande volume absoluto de internações. Por outro lado, municípios menores podem apresentar maior frequência de internações em relação à população residente.

A análise entre 2021 e 2025 também mostra que algumas taxas elevadas se repetem ao longo dos anos, enquanto outras apresentam maior variação. Por isso, volume absoluto, taxa populacional e comportamento temporal devem ser analisados em conjunto.

## 11. Fluxo de pacientes e dependência hospitalar

O município de residência do paciente (`MUNIC_RES`) é comparado ao município onde ocorreu a internação (`MUNIC_MOV`).

Essa comparação permite identificar deslocamentos para atendimento hospitalar e medir a dependência dos municípios em relação à assistência prestada fora do município de residência.

Para essa análise é utilizado inicialmente o ano de 2025, último ano completo disponível.

### 11.1. Fluxo entre município de residência e município de atendimento

In [53]:
fluxo_2025 = (
    demanda_residentes_go[
        demanda_residentes_go["ANO_INTER"] == 2025
    ]
    .groupby(
        [
            "MUNIC_RES",
            "MUNIC_MOV"
        ]
    )
    .agg(
        internacoes=("N_AIH", "size")
    )
    .reset_index()
)

print(
    "Internações analisadas:",
    fluxo_2025["internacoes"].sum()
)

print(
    "Fluxos município-município:",
    len(fluxo_2025)
)

fluxo_2025.head()

Internações analisadas: 455189
Fluxos município-município: 4078


,MUNIC_RES,MUNIC_MOV,internacoes
0,520005,520110,14
1,520005,520140,99
2,520005,520170,1
3,520005,520510,2
4,520005,520540,4


In [54]:
mapa_municipios_go = (
    populacao_go[
        [
            "codigo_sus_6",
            "municipio"
        ]
    ]
    .drop_duplicates()
    .set_index("codigo_sus_6")["municipio"]
)

fluxo_2025["municipio_residencia"] = (
    fluxo_2025["MUNIC_RES"]
    .map(mapa_municipios_go)
)

fluxo_2025["municipio_atendimento"] = (
    fluxo_2025["MUNIC_MOV"]
    .map(mapa_municipios_go)
)

fluxo_2025.head()

,MUNIC_RES,MUNIC_MOV,internacoes,municipio_residencia,municipio_atendimento
0,520005,520110,14,Abadia de Goiás,Anápolis
1,520005,520140,99,Abadia de Goiás,Aparecida de Goiânia
2,520005,520170,1,Abadia de Goiás,Aragarças
3,520005,520510,2,Abadia de Goiás,Catalão
4,520005,520540,4,Abadia de Goiás,Ceres


### 11.2. Dependência de atendimento fora do município

Para cada município de residência, é calculada a proporção de internações realizadas em outro município.

O indicador representa dependência de atendimento hospitalar fora do município de residência e não implica, isoladamente, insuficiência da rede local, pois o deslocamento também pode estar relacionado à regionalização e à concentração de serviços especializados.

In [55]:
demanda_2025_fluxo = (
    demanda_residentes_go[
        demanda_residentes_go["ANO_INTER"] == 2025
    ]
    .copy()
)

demanda_2025_fluxo["fora_municipio"] = (
    demanda_2025_fluxo["MUNIC_RES"]
    != demanda_2025_fluxo["MUNIC_MOV"]
)

dependencia_municipal_2025 = (
    demanda_2025_fluxo
    .groupby("MUNIC_RES")
    .agg(
        internacoes_total=("N_AIH", "size"),
        internacoes_fora=(
            "fora_municipio",
            "sum"
        )
    )
    .reset_index()
)

dependencia_municipal_2025["percentual_fora"] = (
    dependencia_municipal_2025["internacoes_fora"]
    / dependencia_municipal_2025["internacoes_total"]
    * 100
)

dependencia_municipal_2025["municipio"] = (
    dependencia_municipal_2025["MUNIC_RES"]
    .map(mapa_municipios_go)
)

dependencia_municipal_2025 = (
    dependencia_municipal_2025
    .sort_values(
        "percentual_fora",
        ascending=False
    )
    .reset_index(drop=True)
)

dependencia_municipal_2025[
    [
        "municipio",
        "internacoes_total",
        "internacoes_fora",
        "percentual_fora"
    ]
].head(10)

,municipio,internacoes_total,internacoes_fora,percentual_fora
0,Abadia de Goiás,1386,1386,100.0
1,Abadiânia,844,844,100.0
2,Adelândia,135,135,100.0
3,Água Fria de Goiás,124,124,100.0
4,Água Limpa,104,104,100.0
5,Alto Horizonte,503,503,100.0
6,Aporé,151,151,100.0
7,Amaralina,215,215,100.0
8,Aparecida do Rio Doce,187,187,100.0
9,Anhanguera,50,50,100.0


In [56]:
print( "Municípios analisados:", dependencia_municipal_2025["MUNIC_RES"].nunique())
print( "Internações totais:", dependencia_municipal_2025["internacoes_total"].sum())

Municípios analisados: 246
Internações totais: 455189


## 12. Hospitais e municípios absorvedores de demanda

Além do deslocamento dos pacientes, é analisado quais municípios e hospitais recebem maior quantidade de pacientes residentes em outros municípios.

Esses locais podem atuar como polos de atendimento da rede hospitalar, concentrando demanda proveniente de diferentes municípios.

Nesta etapa são consideradas todas as internações atendidas em Goiás em 2025, incluindo pacientes residentes em outros estados. Assim, uma internação externa corresponde a um município de residência diferente do município de atendimento.

### 12.1. Municípios absorvedores

In [57]:
atendimento_2025 = (
    df_sih_demanda[
        df_sih_demanda["ANO_INTER"] == 2025
    ]
    .copy()
)

atendimento_2025["MUNIC_RES"] = (
    atendimento_2025["MUNIC_RES"]
    .astype("string")
    .str.zfill(6)
)

atendimento_2025["MUNIC_MOV"] = (
    atendimento_2025["MUNIC_MOV"]
    .astype("string")
    .str.zfill(6)
)

atendimento_2025["paciente_externo"] = (
    atendimento_2025["MUNIC_RES"]
    != atendimento_2025["MUNIC_MOV"]
)

municipios_absorvedores_2025 = (
    atendimento_2025
    .groupby("MUNIC_MOV")
    .agg(
        internacoes_atendidas=("N_AIH", "size"),
        internacoes_externas=(
            "paciente_externo",
            "sum"
        )
    )
    .reset_index()
)

municipios_absorvedores_2025[
    "percentual_externo"
] = (
    municipios_absorvedores_2025[
        "internacoes_externas"
    ]
    / municipios_absorvedores_2025[
        "internacoes_atendidas"
    ]
    * 100
)

municipios_absorvedores_2025["municipio"] = (
    municipios_absorvedores_2025["MUNIC_MOV"]
    .map(mapa_municipios_go)
)

municipios_absorvedores_2025 = (
    municipios_absorvedores_2025
    .sort_values(
        "internacoes_externas",
        ascending=False
    )
    .reset_index(drop=True)
)

municipios_absorvedores_2025[
    [
        "municipio",
        "internacoes_atendidas",
        "internacoes_externas",
        "percentual_externo"
    ]
].head(10)

,municipio,internacoes_atendidas,internacoes_externas,percentual_externo
0,Goiânia,166721,92794,55.658255
1,Uruaçu,18679,13580,72.701965
2,Aparecida de Goiânia,29046,9251,31.84948
3,Anápolis,28890,7457,25.8117
4,Ceres,8014,6360,79.361118
5,Santa Helena de Goiás,8195,5187,63.294692
6,Itumbiara,12109,4582,37.839623
7,São Luís de Montes Belos,5147,3283,63.784729
8,Trindade,6760,3184,47.100592
9,Catalão,6844,2286,33.40152


### 12.2. Hospitais absorvedores

In [58]:
hospitais_absorvedores_2025 = (
    atendimento_2025
    .groupby(
        [
            "CNES",
            "MUNIC_MOV"
        ]
    )
    .agg(
        internacoes_atendidas=("N_AIH", "size"),
        internacoes_externas=(
            "paciente_externo",
            "sum"
        )
    )
    .reset_index()
)

hospitais_absorvedores_2025[
    "percentual_externo"
] = (
    hospitais_absorvedores_2025[
        "internacoes_externas"
    ]
    / hospitais_absorvedores_2025[
        "internacoes_atendidas"
    ]
    * 100
)

hospitais_absorvedores_2025["municipio"] = (
    hospitais_absorvedores_2025["MUNIC_MOV"]
    .map(mapa_municipios_go)
)

hospitais_absorvedores_2025 = (
    hospitais_absorvedores_2025
    .sort_values(
        "internacoes_externas",
        ascending=False
    )
    .reset_index(drop=True)
)

hospitais_absorvedores_2025[
    [
        "CNES",
        "municipio",
        "internacoes_atendidas",
        "internacoes_externas",
        "percentual_externo"
    ]
].head(10)

,CNES,municipio,internacoes_atendidas,internacoes_externas,percentual_externo
0,7743068,Goiânia,29165,17372,59.564547
1,0547484,Uruaçu,18322,13447,73.392643
2,2338262,Goiânia,18534,10207,55.07176
3,2506815,Goiânia,12214,8439,69.092844
4,2673932,Goiânia,10532,8438,80.117736
5,2339196,Goiânia,9962,8046,80.766914
6,0965324,Goiânia,9939,7405,74.504477
7,2338734,Goiânia,7554,5880,77.839555
8,6665322,Santa Helena de Goiás,6012,5135,85.412508
9,2337517,Ceres,5405,4686,86.697502


## 13. Capacidade × demanda hospitalar

Nesta etapa, a demanda hospitalar é comparada à capacidade de leitos SUS registrada no CNES.

Como os dados do CNES representam fotografias mensais da capacidade instalada, os leitos não são somados ao longo do ano. Para 2025 é utilizada a média mensal de leitos SUS de cada município.

A relação entre internações e leitos é utilizada como indicador de intensidade de utilização da estrutura disponível. Esse indicador não corresponde diretamente à taxa de ocupação hospitalar.

### 13.1. Capacidade mensal por município

In [59]:
df_cnes["CODUFMUN"] = (
    df_cnes["CODUFMUN"]
    .astype("string")
    .str.zfill(6)
)

In [60]:
capacidade_municipal_mes_2025 = (
    df_cnes[
        df_cnes["ANO"] == 2025
    ]
    .groupby(
        [
            "CODUFMUN",
            "MES"
        ]
    )
    .agg(
        leitos_sus=("QT_SUS_NUM", "sum"),
        leitos_existentes=("QT_EXIST_NUM", "sum")
    )
    .reset_index()
)

capacidade_municipal_mes_2025.head()

,CODUFMUN,MES,leitos_sus,leitos_existentes
0,520010,1,9,9
1,520010,2,9,9
2,520010,3,9,9
3,520010,4,9,9
4,520010,5,9,9


In [61]:
capacidade_municipal_2025 = (
    capacidade_municipal_mes_2025
    .groupby("CODUFMUN")
    .agg(
        meses_cnes=("MES", "nunique"),
        leitos_sus_medio=("leitos_sus", "mean"),
        leitos_existentes_medio=(
            "leitos_existentes",
            "mean"
        )
    )
    .reset_index()
)

capacidade_municipal_2025.head()

,CODUFMUN,meses_cnes,leitos_sus_medio,leitos_existentes_medio
0,520010,12,9.0,9.0
1,520013,12,37.25,72.25
2,520015,12,15.0,15.0
3,520017,4,10.0,10.0
4,520025,12,157.0,206.0


### 13.2. Demanda por município de atendimento

In [62]:
demanda_atendimento_2025 = (
    atendimento_2025
    .groupby("MUNIC_MOV")
    .agg(
        internacoes=("N_AIH", "size")
    )
    .reset_index()
)

demanda_atendimento_2025.head()

,MUNIC_MOV,internacoes
0,520013,356
1,520025,4564
2,520030,2318
3,520050,11
4,520060,346


In [63]:
capacidade_demanda_2025 = (
    capacidade_municipal_2025
    .merge(
        demanda_atendimento_2025,
        left_on="CODUFMUN",
        right_on="MUNIC_MOV",
        how="outer"
    )
)

capacidade_demanda_2025["internacoes"] = (
    capacidade_demanda_2025["internacoes"]
    .fillna(0)
    .astype(int)
)

capacidade_demanda_2025["codigo_municipio"] = (
    capacidade_demanda_2025["CODUFMUN"]
    .fillna(
        capacidade_demanda_2025["MUNIC_MOV"]
    )
)

capacidade_demanda_2025["municipio"] = (
    capacidade_demanda_2025["codigo_municipio"]
    .map(mapa_municipios_go)
)

In [64]:
capacidade_demanda_2025[
    "internacoes_por_leito_sus_medio"
] = (
    capacidade_demanda_2025["internacoes"]
    / capacidade_demanda_2025[
        "leitos_sus_medio"
    ].replace(0, pd.NA)
)

capacidade_demanda_2025 = (
    capacidade_demanda_2025
    .sort_values(
        "internacoes_por_leito_sus_medio",
        ascending=False
    )
    .reset_index(drop=True)
)

capacidade_demanda_2025[
    [
        "municipio",
        "internacoes",
        "leitos_sus_medio",
        "leitos_existentes_medio",
        "internacoes_por_leito_sus_medio",
        "meses_cnes"
    ]
].head(10)

,municipio,internacoes,leitos_sus_medio,leitos_existentes_medio,internacoes_por_leito_sus_medio,meses_cnes
0,São Miguel do Araguaia,5962,59.0,100.0,101.050847,12
1,Pirenópolis,3430,34.0,34.0,100.882353,12
2,Formosa,8089,86.75,163.083333,93.244957,12
3,São Luís de Montes Belos,5147,58.666667,85.333333,87.732955,12
4,Jaraguá,4985,62.0,80.0,80.403226,12
5,Bom Jesus de Goiás,2200,28.0,28.0,78.571429,12
6,Itaberaí,1940,27.0,65.0,71.851852,12
7,Alexânia,2318,34.083333,34.083333,68.00978,12
8,Silvânia,2974,44.0,44.0,67.590909,12
9,Quirinópolis,4240,63.0,138.333333,67.301587,12


In [65]:
print(
    "Municípios com média de leitos SUS igual a zero:",
    (
        capacidade_demanda_2025[
            "leitos_sus_medio"
        ] == 0
    ).sum()
)

Municípios com média de leitos SUS igual a zero: 0


In [66]:
print( "Municípios na análise:", capacidade_demanda_2025[ "codigo_municipio"].nunique())
print( "Internações consideradas:", capacidade_demanda_2025[ "internacoes"].sum())

print(
    "Municípios com 12 meses de CNES:",
    (
        capacidade_demanda_2025[
            "meses_cnes"
        ] == 12
    ).sum()
)

print(
    "Municípios sem informação de leitos SUS:",
    capacidade_demanda_2025[
        "leitos_sus_medio"
    ].isna().sum()
)

Municípios na análise: 197
Internações consideradas: 459420
Municípios com 12 meses de CNES: 193
Municípios sem informação de leitos SUS: 0


### 13.3. Validação da cobertura mensal da capacidade

Para a comparação entre demanda anual e capacidade média de leitos, são considerados no ranking principal apenas os municípios que possuem os 12 meses de dados do CNES em 2025.

Municípios com cobertura parcial permanecem identificados na base, mas não são utilizados na comparação principal, pois a média de leitos calculada com poucos meses pode não representar adequadamente a capacidade do ano.

In [67]:
municipios_cobertura_parcial_2025 = (
    capacidade_demanda_2025[
        capacidade_demanda_2025["meses_cnes"] < 12
    ][
        [
            "municipio",
            "meses_cnes",
            "internacoes",
            "leitos_sus_medio"
        ]
    ]
    .sort_values("meses_cnes")
)

municipios_cobertura_parcial_2025

,municipio,meses_cnes,internacoes,leitos_sus_medio
195,Terezópolis de Goiás,2,0,3.0
180,Água Fria de Goiás,4,0,10.0
187,Lagoa Santa,5,0,3.0
175,Aparecida do Rio Doce,6,0,11.5


In [68]:
capacidade_demanda_2025_validada = (
    capacidade_demanda_2025[
        capacidade_demanda_2025["meses_cnes"] == 12
    ]
    .copy()
)

print(
    "Municípios com cobertura completa:",
    capacidade_demanda_2025_validada[
        "codigo_municipio"
    ].nunique()
)

print(
    "Internações nesses municípios:",
    capacidade_demanda_2025_validada[
        "internacoes"
    ].sum()
)

Municípios com cobertura completa: 193
Internações nesses municípios: 459420


In [69]:
capacidade_demanda_2025_validada[
    [
        "municipio",
        "internacoes",
        "leitos_sus_medio",
        "leitos_existentes_medio",
        "internacoes_por_leito_sus_medio",
        "meses_cnes"
    ]
].sort_values(
    "internacoes_por_leito_sus_medio",
    ascending=False
).head(10)

,municipio,internacoes,leitos_sus_medio,leitos_existentes_medio,internacoes_por_leito_sus_medio,meses_cnes
0,São Miguel do Araguaia,5962,59.0,100.0,101.050847,12
1,Pirenópolis,3430,34.0,34.0,100.882353,12
2,Formosa,8089,86.75,163.083333,93.244957,12
3,São Luís de Montes Belos,5147,58.666667,85.333333,87.732955,12
4,Jaraguá,4985,62.0,80.0,80.403226,12
5,Bom Jesus de Goiás,2200,28.0,28.0,78.571429,12
6,Itaberaí,1940,27.0,65.0,71.851852,12
7,Alexânia,2318,34.083333,34.083333,68.00978,12
8,Silvânia,2974,44.0,44.0,67.590909,12
9,Quirinópolis,4240,63.0,138.333333,67.301587,12


## 14. Permanência hospitalar

A permanência hospitalar é analisada a partir do número de dias registrado em cada internação.

Para esta etapa são consideradas as internações regulares de 2025. Além das medidas gerais de permanência, é construída uma referência interna para grupos comparáveis de internações, considerando o procedimento realizado e a especialidade.

A referência utilizada corresponde ao percentil 75 da permanência observada em cada grupo e não representa um parâmetro clínico oficial.

Alguns grupos possuem baixo número de internações e, nesses casos, a referência deve ser interpretada com cautela devido à menor estabilidade estatística.

In [70]:
permanencia_2025 = (
    df_sih_demanda[
        df_sih_demanda["ANO_INTER"] == 2025
    ]
    .copy()
)

permanencia_2025["DIAS_PERM_NUM"] = pd.to_numeric(
    permanencia_2025["DIAS_PERM"],
    errors="coerce"
)

resumo_permanencia_2025 = pd.DataFrame(
    {
        "internacoes": [
            len(permanencia_2025)
        ],
        "media_dias": [
            permanencia_2025["DIAS_PERM_NUM"].mean()
        ],
        "mediana_dias": [
            permanencia_2025["DIAS_PERM_NUM"].median()
        ],
        "p75_dias": [
            permanencia_2025["DIAS_PERM_NUM"].quantile(0.75)
        ],
        "p90_dias": [
            permanencia_2025["DIAS_PERM_NUM"].quantile(0.90)
        ],
        "maximo_dias": [
            permanencia_2025["DIAS_PERM_NUM"].max()
        ]
    }
)

resumo_permanencia_2025

,internacoes,media_dias,mediana_dias,p75_dias,p90_dias,maximo_dias
0,459420,4.015813,2.0,4.0,9.0,337


In [71]:
permanencia_2025["referencia_p75_dias"] = (
    permanencia_2025
    .groupby(
        [
            "PROC_REA",
            "ESPEC"
        ],
        dropna=False
    )["DIAS_PERM_NUM"]
    .transform(
        lambda x: x.quantile(0.75)
    )
)

permanencia_2025["acima_referencia"] = (
    permanencia_2025["DIAS_PERM_NUM"]
    > permanencia_2025["referencia_p75_dias"]
)

permanencia_2025["dias_acima_referencia"] = (
    permanencia_2025["DIAS_PERM_NUM"]
    - permanencia_2025["referencia_p75_dias"]
).clip(lower=0)

print(
    "Internações sem referência:",
    permanencia_2025[
        "referencia_p75_dias"
    ].isna().sum()
)

print(
    "Internações acima da referência:",
    permanencia_2025[
        "acima_referencia"
    ].sum()
)

print(
    "Percentual acima da referência:",
    round(
        permanencia_2025[
            "acima_referencia"
        ].mean() * 100,
        2
    ),
    "%"
)

Internações sem referência: 0
Internações acima da referência: 84364
Percentual acima da referência: 18.36 %


In [72]:
tamanho_grupos_referencia = (
    permanencia_2025
    .groupby(
        [
            "PROC_REA",
            "ESPEC"
        ],
        dropna=False
    )
    .size()
)

print(
    "Grupos de referência:",
    len(tamanho_grupos_referencia)
)

print(
    "Menor tamanho de grupo:",
    tamanho_grupos_referencia.min()
)

print(
    "Mediana do tamanho dos grupos:",
    tamanho_grupos_referencia.median()
)

print(
    "Grupos com apenas 1 internação:",
    (tamanho_grupos_referencia == 1).sum()
)

Grupos de referência: 1810
Menor tamanho de grupo: 1
Mediana do tamanho dos grupos: 12.0
Grupos com apenas 1 internação: 271


## 15. Dias acima da referência e valor registrado nas AIHs

As internações com permanência acima da referência observada para grupos comparáveis são analisadas em relação ao número de dias excedentes e ao valor registrado na AIH.

O campo `VAL_TOT` representa o valor registrado/aprovado da AIH e não deve ser interpretado como custo real da internação.

O valor financeiro é apresentado como valor associado às AIHs classificadas com permanência acima da referência, sem atribuir diretamente esse valor aos dias excedentes.

In [73]:
permanencia_2025["VAL_TOT_NUM"] = pd.to_numeric(
    permanencia_2025["VAL_TOT"],
    errors="coerce"
)

internacoes_acima_2025 = (
    permanencia_2025[
        permanencia_2025["acima_referencia"]
    ]
    .copy()
)

resumo_acima_referencia_2025 = pd.DataFrame(
    {
        "internacoes_acima_referencia": [
            len(internacoes_acima_2025)
        ],
        "dias_acima_referencia": [
            internacoes_acima_2025[
                "dias_acima_referencia"
            ].sum()
        ],
        "valor_aih_registrado": [
            internacoes_acima_2025[
                "VAL_TOT_NUM"
            ].sum()
        ]
    }
)

resumo_acima_referencia_2025

,internacoes_acima_referencia,dias_acima_referencia,valor_aih_registrado
0,84364,472068.0,2.681352e+08


In [74]:
permanencia_acima_municipio_2025 = (
    internacoes_acima_2025
    .groupby("MUNIC_MOV")
    .agg(
        internacoes_acima=(
            "N_AIH",
            "size"
        ),
        dias_acima_referencia=(
            "dias_acima_referencia",
            "sum"
        ),
        valor_aih_registrado=(
            "VAL_TOT_NUM",
            "sum"
        )
    )
    .reset_index()
)

permanencia_acima_municipio_2025["municipio"] = (
    permanencia_acima_municipio_2025[
        "MUNIC_MOV"
    ]
    .map(mapa_municipios_go)
)

permanencia_acima_municipio_2025 = (
    permanencia_acima_municipio_2025
    .sort_values(
        "dias_acima_referencia",
        ascending=False
    )
    .reset_index(drop=True)
)

permanencia_acima_municipio_2025[
    [
        "municipio",
        "internacoes_acima",
        "dias_acima_referencia",
        "valor_aih_registrado"
    ]
].head(10)

,municipio,internacoes_acima,dias_acima_referencia,valor_aih_registrado
0,Goiânia,36723,235780.25,148689044.05
1,Aparecida de Goiânia,6019,33535.0,22982744.2
2,Uruaçu,5606,30525.0,12605118.49
3,Anápolis,5434,27604.5,21826967.07
4,Rio Verde,3663,21777.75,9355165.35
5,Itumbiara,2714,15832.5,7825258.01
6,Nerópolis,1306,8364.5,9476449.72
7,Jataí,1270,7542.5,3269788.3
8,Santa Helena de Goiás,1403,7494.75,4051540.42
9,Formosa,1351,6045.75,2653903.32


## 16. Pressão hospitalar mensal

A pressão hospitalar é analisada mensalmente relacionando o número de internações atendidas à quantidade de leitos SUS disponível no mesmo município e mês.

O indicador de internações por leito SUS representa uma medida de intensidade de utilização da capacidade disponível, não uma taxa de ocupação hospitalar.

Para garantir comparabilidade, são utilizados apenas municípios com os 12 meses de dados do CNES em 2025.

In [75]:
demanda_municipal_mes_2025 = (
    atendimento_2025
    .groupby(
        [
            "MUNIC_MOV",
            "MES_INTER"
        ]
    )
    .agg(
        internacoes=("N_AIH", "size")
    )
    .reset_index()
    .rename(
        columns={
            "MES_INTER": "MES"
        }
    )
)

demanda_municipal_mes_2025["MES"] = ( demanda_municipal_mes_2025["MES"].astype(int))
capacidade_municipal_mes_2025["MES"] = ( capacidade_municipal_mes_2025["MES"].astype(int))

In [76]:
codigos_cobertura_completa = set(
    capacidade_demanda_2025_validada[
        "codigo_municipio"
    ]
)

pressao_mensal_2025 = (
    capacidade_municipal_mes_2025[
        capacidade_municipal_mes_2025[
            "CODUFMUN"
        ].isin(codigos_cobertura_completa)
    ]
    .merge(
        demanda_municipal_mes_2025,
        left_on=[
            "CODUFMUN",
            "MES"
        ],
        right_on=[
            "MUNIC_MOV",
            "MES"
        ],
        how="left",
        validate="one_to_one"
    )
)

pressao_mensal_2025["internacoes"] = (
    pressao_mensal_2025["internacoes"]
    .fillna(0)
    .astype(int)
)

pressao_mensal_2025[
    "internacoes_por_leito_sus"
] = (
    pressao_mensal_2025["internacoes"]
    / pressao_mensal_2025[
        "leitos_sus"
    ].replace(0, pd.NA)
)

In [77]:
pressao_mensal_2025[
    "mediana_municipio"
] = (
    pressao_mensal_2025
    .groupby("CODUFMUN")[
        "internacoes_por_leito_sus"
    ]
    .transform("median")
)

pressao_mensal_2025[
    "p75_municipio"
] = (
    pressao_mensal_2025
    .groupby("CODUFMUN")[
        "internacoes_por_leito_sus"
    ]
    .transform(
        lambda x: x.quantile(0.75)
    )
)

pressao_mensal_2025[
    "indice_pressao_relativa"
] = (
    pressao_mensal_2025[
        "internacoes_por_leito_sus"
    ]
    / pressao_mensal_2025[
        "mediana_municipio"
    ].replace(0, pd.NA)
)

pressao_mensal_2025[
    "acima_p75_municipio"
] = (
    pressao_mensal_2025[
        "internacoes_por_leito_sus"
    ]
    > pressao_mensal_2025[
        "p75_municipio"
    ]
)

pressao_mensal_2025["municipio"] = (
    pressao_mensal_2025["CODUFMUN"]
    .map(mapa_municipios_go)
)

In [78]:
pressao_mensal_2025[
    [
        "municipio",
        "MES",
        "internacoes",
        "leitos_sus",
        "internacoes_por_leito_sus",
        "indice_pressao_relativa",
        "acima_p75_municipio"
    ]
].sort_values(
    "internacoes_por_leito_sus",
    ascending=False
).head(15)

,municipio,MES,internacoes,leitos_sus,internacoes_por_leito_sus,indice_pressao_relativa,acima_p75_municipio
2066,São Luís de Montes Belos,3,493,52,9.480769,1.391261,True
1756,Pirenópolis,5,311,34,9.147059,1.108734,True
1757,Pirenópolis,6,311,34,9.147059,1.108734,True
1755,Pirenópolis,4,307,34,9.029412,1.094474,True
1761,Pirenópolis,10,302,34,8.882353,1.076649,False
1762,Pirenópolis,11,298,34,8.764706,1.062389,False
2064,São Luís de Montes Belos,1,447,52,8.596154,1.261447,True
848,Formosa,9,713,83,8.590361,1.102862,True
2096,São Miguel do Araguaia,9,504,59,8.542373,1.012048,True
2089,São Miguel do Araguaia,2,504,59,8.542373,1.012048,True


In [79]:
pressao_go_mensal_2025_validada = (
    pressao_mensal_2025
    .groupby("MES")
    .agg(
        internacoes=("internacoes", "sum"),
        leitos_sus=("leitos_sus", "sum")
    )
    .reset_index()
)

pressao_go_mensal_2025_validada[
    "internacoes_por_leito_sus"
] = (
    pressao_go_mensal_2025_validada["internacoes"]
    / pressao_go_mensal_2025_validada["leitos_sus"]
)

pressao_go_mensal_2025_validada

,MES,internacoes,leitos_sus,internacoes_por_leito_sus
0,1,37301,13417,2.78013
1,2,36292,13392,2.709976
2,3,39206,13413,2.922985
3,4,39213,13445,2.916549
4,5,40583,13540,2.997267
5,6,38121,13529,2.817725
6,7,38076,13566,2.806723
7,8,37951,13634,2.783556
8,9,39043,13615,2.867646
9,10,39562,13475,2.935955


### 16.1. Síntese da pressão hospitalar em 2025

No conjunto de municípios com cobertura completa do CNES em 2025, a relação mensal entre internações e leitos SUS apresentou variação relativamente pequena ao longo do ano.

O maior valor foi observado em maio, com aproximadamente 3,00 internações por leito SUS, enquanto fevereiro apresentou o menor valor, com aproximadamente 2,71.

Esses resultados representam a intensidade mensal de utilização da capacidade hospitalar disponível e não correspondem a uma taxa de ocupação hospitalar.

## 17. Conclusão

A análise integrada permitiu relacionar internações hospitalares, capacidade de leitos e população municipal para identificar padrões de demanda e utilização da rede hospitalar de Goiás.

As análises mostraram diferenças entre volume absoluto de internações e demanda relativa à população, além de deslocamentos de pacientes entre municípios e concentração de atendimentos em determinados municípios e estabelecimentos.

A comparação entre demanda e capacidade foi realizada utilizando os registros mensais de leitos do CNES, sendo os indicadores de internações por leito interpretados como medidas de intensidade de utilização e não como taxas de ocupação hospitalar.

A análise de permanência utilizou uma referência interna baseada no percentil 75 de grupos comparáveis de procedimento e especialidade. Os valores associados às AIHs acima dessa referência representam valores registrados/aprovados e não custos reais dos dias adicionais.

Para 2026, os resultados de internações representam apenas o período de janeiro a junho. Para 2023, os indicadores populacionais utilizam uma população de referência derivada por interpolação entre 2022 e 2024.